In [1]:
quiet_library <- function(...) { suppressPackageStartupMessages(library(...)) }
quiet_library(hise)
quiet_library(dplyr)
quiet_library(purrr)

Warning message:
“package ‘dplyr’ was built under R version 4.4.3”
Warning message:
“package ‘purrr’ was built under R version 4.4.3”


## Read GSEA results stored in HISE

In [2]:
gsea_uuid <- list("ad8d7320-399c-41ff-8b2f-b840d8eae324")

In [3]:
gsea_file <- cacheFiles(gsea_uuid)

[2026-10-28 19:48:38] INFO  Calling cacheFiles
[1] "downloading fileID ad8d7320-399c-41ff-8b2f-b840d8eae324"
[2026-10-28 19:48:44] INFO  Finished cacheFiles successfully (time_elapsed=2.504s)


In [4]:
no_psm_gsea_uuid <- list("b947b734-04aa-41af-9019-d97e20437af9")

In [5]:
no_psm_file <- cacheFiles(no_psm_gsea_uuid)

[2026-10-28 19:48:44] INFO  Calling cacheFiles
[1] "downloading fileID b947b734-04aa-41af-9019-d97e20437af9"
[2026-10-28 19:48:48] INFO  Finished cacheFiles successfully (time_elapsed=1.985s)


In [8]:
gsea <- read.table(gsea_file, sep = "\t", quote = "", header = TRUE)
no_psm_gsea <- read.table(no_psm_file, sep = "\t", quote = "", header = TRUE)

In [9]:
names(no_psm_gsea)

[1] "fg"              "bg"              "timepoint"       "aifi_cell_type" 
 [5] "id"              "pathway"         "NES"             "nomP"           
 [9] "adjP"            "n_leadingEdge"   "n_pathway_genes" "leadingEdge"    
[13] "pathway_genes"

In [10]:
no_psm_gsea <- no_psm_gsea %>%
  mutate(pathway = paste("No PSM Genes |",pathway))

In [11]:
all_gsea <- rbind(gsea, no_psm_gsea)

## Format for output

In [12]:
names(all_gsea)

[1] "fg"              "bg"              "timepoint"       "aifi_cell_type" 
 [5] "id"              "pathway"         "NES"             "nomP"           
 [9] "adjP"            "n_leadingEdge"   "n_pathway_genes" "leadingEdge"    
[13] "pathway_genes"

In [13]:
all_gsea <- all_gsea %>%
  rename(pathway_label = pathway,
         pathway = id) %>%
  mutate(pathway_label = gsub("'","",pathway_label)) %>%
  select(-pathway_genes) %>%
  select(aifi_cell_type, timepoint, fg, bg,
         pathway, pathway_label, n_pathway_genes,
         NES, nomP, adjP,
         n_leadingEdge, leadingEdge)

In [14]:
out_dir <- "output"
if(!dir.exists("output")) { dir.create("output") }

In [15]:
out_file <- paste0("Supp-Table-07_Reactome_GSEA_", Sys.Date(), ".tsv")
write.table(
    all_gsea,
    file.path(out_dir, out_file),
    row.names = FALSE,
    quote = FALSE,
    sep = "\t"
)

In [16]:
system(paste('gzip', file.path(out_dir, out_file)))

In [17]:
sessionInfo()

R version 4.4.1 (2024-06-14)
Platform: x86_64-conda-linux-gnu
Running under: Ubuntu 22.04.5 LTS

Matrix products: default
BLAS/LAPACK: /home/workspace/environment/archrpixiv12/.pixi/envs/default/lib/libopenblasp-r0.3.33.so;  LAPACK version 3.12.0

locale:
 [1] LC_CTYPE=C.UTF-8    LC_NUMERIC=C        LC_TIME=C          
 [4] LC_COLLATE=C        LC_MONETARY=C       LC_MESSAGES=C      
 [7] LC_PAPER=C          LC_NAME=C           LC_ADDRESS=C       
[10] LC_TELEPHONE=C      LC_MEASUREMENT=C    LC_IDENTIFICATION=C

time zone: America/Los_Angeles
tzcode source: system (glibc)

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
[1] purrr_1.2.2 dplyr_1.2.1 hise_3.6.15

loaded via a namespace (and not attached):
 [1] crayon_1.5.3      vctrs_0.7.3       httr_1.4.8        cli_3.6.6        
 [5] rlang_1.2.0       generics_0.1.4    textshaping_0.4.0 assertthat_0.2.1 
 [9] jsonlite_2.0.0    glue_1.8.1        RCurl_1.98-1.19   